In [1]:
# imports
import re
from gensim.models import Word2Vec
import nltk
from nltk.corpus import stopwords
en_stopwords = stopwords.words('english')
es_stopwords = stopwords.words('spanish')
de_stopwords = stopwords.words('german')
tokenizer = nltk.tokenize.RegexpTokenizer(r'\w+')


In [2]:
# load wikipedia data for education
def load_text_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

texts = {
    "en": load_text_file("../data/Education/English_education.txt"),
    "es": load_text_file("../data/Education/Spanish_education.txt"),
    "de": load_text_file("../data/Education/German_education.txt")
}

stopwords = {
    "en": en_stopwords,
    "es": es_stopwords,
    "de": de_stopwords
}

In [4]:
# preprocess and tokenize
def preprocess(text, lang):
    text = text.lower()
    # text = re.sub(r'[^a-zA-Záéíóúüñßäö\s]', ' ', text)
    # text = re.sub(r'\s+', ' ', text).strip()
    tokens = tokenizer.tokenize(text)
    return [i for i in tokens if i not in stopwords[lang]]
    

tokenized = {
    lang: preprocess(text, lang)
    for lang, text in texts.items()
}

In [5]:
# information about each of the wikipedia pages

stats = {}
for lang, tokens in tokenized.items():
    stats[lang] = {
        "num_words": len(tokens),
        "vocab_size": len(set(tokens))
    }

total_tokens = sum(s["num_words"] for s in stats.values())

print("Per-language stats:")
for lang, s in stats.items():
    print(lang, s)

print("Total tokens across all datasets:", total_tokens)


Per-language stats:
en {'num_words': 5629, 'vocab_size': 1824}
es {'num_words': 2869, 'vocab_size': 1474}
de {'num_words': 596, 'vocab_size': 436}
Total tokens across all datasets: 9094


In [6]:
# train word2vec modelss
models = {}

for lang, tokens in tokenized.items():
    
    # split into pseudo-sentences (since Wikipedia text is continuous)
    sentences = [tokens[i:i+50] for i in range(0, len(tokens), 50)]
    
    model = Word2Vec(
        sentences=sentences,
        vector_size=100,
        window=5,
        min_count=2,
        workers=4
    )
    
    models[lang] = model

In [7]:
print("English closest words to 'education':")
print(models["en"].wv.most_similar("education", topn=5))

print("Spanish closest words to 'education':")
print(models["es"].wv.most_similar("educación", topn=5))

print("German closest words to 'education':")
print(models["de"].wv.most_similar("ausbildung", topn=5))

English closest words to 'education':
[('learning', 0.8190154433250427), ('students', 0.8170468807220459), ('educational', 0.8000394701957703), ('also', 0.7963951826095581), ('like', 0.7940818667411804)]
Spanish closest words to 'education':
[('etapa', 0.4229865074157715), ('ambiente', 0.378479540348053), ('instituciones', 0.33635804057121277), ('gran', 0.31782612204551697), ('nivel', 0.3138487935066223)]
German closest words to 'education':
[('erzieher', 0.2989494204521179), ('entweder', 0.22514817118644714), ('stunden', 0.21436665952205658), ('mehr', 0.2097112238407135), ('wurde', 0.2025429606437683)]
